### Class Imbalance Figure

In [0]:
%sql
SELECT *
FROM multimodal_adni_living_binary

Databricks visualization. Run in Databricks to view.

### Class Imbalance Figure (After binary classification)

In [0]:
%sql
SELECT *
FROM multimodal_adni_living_binary

Databricks visualization. Run in Databricks to view.

### Class Imbalance after changing to 50/50
![image_1773545651518.png](./image_1773545651518.png "image_1773545651518.png")

## Top 8 Features (Importance by SHAP)
- WORD1DL
- WORD3DL
- WORD2DL
- MMDRAW
- MMW
- MMLTR5 (and shows W is extremely popular)
- MMO
- MMREPEAT

In [0]:
%sql
SELECT *
FROM multimodal_adni_living_binary

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%pip install shap
%pip install xgboost
%restart_python

In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import json
import shap
import numpy as np

# Replace Run ID when you want to check a different run, should be at the bottom of the mlflow logging step above
# Before: f0436711bfa24095b86d8df85110c670
# After 2/27 variable change: 36efe52695df49608ece51cebb470c16
RUN_ID = "36efe52695df49608ece51cebb470c16"

# Load model
model = mlflow.xgboost.load_model(f"runs:/{RUN_ID}/model")

# Download artifacts from MLFlow run. Feature_cols, X_test, and y_test
feature_cols_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="feature_columns.json"
)
X_test_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="X_test.parquet"
)
y_test_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="y_test.parquet"
)

with open(feature_cols_path, "r") as f:
    feature_cols = json.load(f)

X_test = pd.read_parquet(X_test_path)[feature_cols]
y_test = pd.read_parquet(y_test_path)["y"].astype(int)

# ---- TreeSHAP ----
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Handle version differences: sometimes list per class (suggested by ai)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, X_test)

## Correlation Matrix
From what I can see of the top 50 SHAP variables (actually 47, 3 were removed since they werent numberic) it shows not too much multicollinearity, although within variables from MM or MOCA are more related, which makes sense as they are from the same test.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SHAP importance table
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# Mean absolute SHAP per feature
mean_abs_shap = np.abs(shap_vals).mean(axis=0)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

# Select top 50 SHAP features
top_n = 50
top_features = importance_df["feature"].head(top_n).tolist()

# Subset data to only these features
df = X_test.copy()

# only allow numeric columns
df = df.select_dtypes(include=[np.number])

# Keep only the top SHAP-ranked features that are present in df
top_features_present = [col for col in top_features if col in df.columns]
df_top = df[top_features_present]

# Compute correlation matrix
corr = df_top.corr(method="spearman") # Spearman is often better when relationships may be monotonic

# Plot
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Spearman correlation")

ax.set_title(f"Correlation Heatmap of Top {len(top_features_present)} SHAP Features")
plt.tight_layout()
plt.show()

## Conclusion
All of this Exploratory Data Analysis has helped me realize why some of these variables contribute so much to the final outcome, and especially answered my question as to why MMLTR5_W (the derived encoded variable) was so specific about W. I still don't know the reason behind it, but it makes much more sense to see now the types of data that were in that feature.

I believe that Test Variables from MMSE and MOCA will be the primary predictors for the binary outcome, especially the predictors that include language. It is extremely interesting to see them in the higher parts of SHAP, and I feel very proud knowing that I have added a small amount of evidence towards the research Dr. Orimaye has already done about linguistics and Alzheimer's. I believe that there will be a big correlation between the target and this specifically.

I predict that Vital Signs will have less influence than what they show in SHAP, but still correlate passively such as with weight and height. It does make me want to create a BMI statistic, so maybe it can be a bit more clear on why certain weights or heights matter in the final output.

I do find it interesting to also see GD here, and that there is a good chance of a link between depression and Alzheimer's as I have read in some of Dr. Orimaye's research papers. I hope to come to find that those factors will come into play when predicting the output variable.